# Customer Churn Prediction: Profit-Based Evaluation of ML Models
## Step 8: Conclusion & Final Summary

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib
from sklearn.metrics import confusion_matrix, roc_auc_score

sns.set_theme(style='whitegrid')

# Load everything
X_test               = pd.read_csv('X_test.csv')
y_test               = pd.read_csv('y_test.csv').squeeze()
traditional_metrics  = pd.read_csv('traditional_metrics.csv',  index_col='Model')
final_comparison     = pd.read_csv('final_comparison.csv',     index_col='Model')

models = {
    'Logistic Regression' : joblib.load('logistic_regression_model.pkl'),
    'Random Forest'       : joblib.load('random_forest_model.pkl'),
    'Gradient Boosting'   : joblib.load('gradient_boosting_model.pkl')
}

print('All data and models loaded ✓')

### 8.1 — Full Results Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Customer Churn Prediction — Full Results Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

colors      = ['#2E86AB', '#E74C3C', '#2ECC71']
model_names = list(models.keys())

# ── Plot 1: Traditional Metrics ───────────────────────────────────────────
ax1  = fig.add_subplot(2, 3, 1)
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
x    = np.arange(len(metrics_to_plot))
w    = 0.25
for i, (name, color) in enumerate(zip(model_names, colors)):
    ax1.bar(x + i*w, traditional_metrics.loc[name, metrics_to_plot],
            w, label=name, color=color, edgecolor='white', alpha=0.9)
ax1.set_title('Traditional Metrics', fontweight='bold')
ax1.set_xticks(x + w)
ax1.set_xticklabels(['Acc', 'Prec', 'Rec', 'F1', 'AUC'], fontsize=9)
ax1.set_ylabel('Score (%)')
ax1.set_ylim(50, 105)
ax1.legend(fontsize=7)

# ── Plot 2: ROC Curves ────────────────────────────────────────────────────
from sklearn.metrics import roc_curve
ax2 = fig.add_subplot(2, 3, 2)
for (name, model), color in zip(models.items(), colors):
    y_proba      = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _  = roc_curve(y_test, y_proba)
    auc          = roc_auc_score(y_test, y_proba)
    ax2.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} ({auc:.3f})')
ax2.plot([0,1],[0,1],'k--', linewidth=1)
ax2.set_title('ROC Curves', fontweight='bold')
ax2.set_xlabel('FPR')
ax2.set_ylabel('TPR')
ax2.legend(fontsize=7)

# ── Plot 3: EMP Comparison ────────────────────────────────────────────────
ax3      = fig.add_subplot(2, 3, 3)
emp_vals = final_comparison['EMP ($)'].values
bars     = ax3.bar(model_names, emp_vals, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, emp_vals):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 300,
             f'${val:,.0f}', ha='center', fontweight='bold', fontsize=9)
ax3.set_title('Expected Maximum Profit', fontweight='bold')
ax3.set_ylabel('EMP ($)')
ax3.set_xticklabels(model_names, fontsize=8, rotation=10)
ax3.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# ── Plot 4: Traditional Ranking ───────────────────────────────────────────
ax4          = fig.add_subplot(2, 3, 4)
medal_colors = ['gold', 'silver', '#cd7f32']
trad_sorted  = final_comparison['ROC-AUC Score (%)'].sort_values(ascending=False)
ax4.bar(range(len(trad_sorted)), trad_sorted.values,
        color=medal_colors, edgecolor='white', width=0.5)
ax4.set_xticks(range(len(trad_sorted)))
ax4.set_xticklabels([f'#{i+1}\n{n}' for i,n in enumerate(trad_sorted.index)], fontsize=8)
ax4.set_title('Ranking by ROC-AUC\n(Traditional)', fontweight='bold')
ax4.set_ylabel('ROC-AUC (%)')
ax4.set_ylim(70, 100)
for i, val in enumerate(trad_sorted.values):
    ax4.text(i, val+0.2, f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

# ── Plot 5: Profit Ranking ────────────────────────────────────────────────
ax5           = fig.add_subplot(2, 3, 5)
profit_sorted = final_comparison['EMP ($)'].sort_values(ascending=False)
ax5.bar(range(len(profit_sorted)), profit_sorted.values,
        color=medal_colors, edgecolor='white', width=0.5)
ax5.set_xticks(range(len(profit_sorted)))
ax5.set_xticklabels([f'#{i+1}\n{n}' for i,n in enumerate(profit_sorted.index)], fontsize=8)
ax5.set_title('Ranking by EMP\n(Profit-Based)', fontweight='bold')
ax5.set_ylabel('EMP ($)')
ax5.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
for i, val in enumerate(profit_sorted.values):
    ax5.text(i, val+300, f'${val:,.0f}', ha='center', fontsize=9, fontweight='bold')

# ── Plot 6: Rank Change Table ─────────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
table_data = [
    [name,
     str(int(final_comparison.loc[name, 'Traditional Rank'])),
     str(int(final_comparison.loc[name, 'Profit Rank'])),
     final_comparison.loc[name, 'Rank Changed?']]
    for name in model_names
]
table = ax6.table(
    cellText   = table_data,
    colLabels  = ['Model', 'Trad. Rank', 'Profit Rank', 'Changed?'],
    loc        = 'center',
    cellLoc    = 'center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 2.2)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1F3864')
        cell.set_text_props(color='white', fontweight='bold')
    elif '⚠' in str(cell.get_text().get_text()):
        cell.set_facecolor('#FADBD8')
ax6.set_title('Ranking Comparison Summary', fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('plot_20_full_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved ✓')

### 8.2 — Written Conclusion

In [ ]:
best_trad   = final_comparison['ROC-AUC Score (%)'].idxmax()
best_profit = final_comparison['EMP ($)'].idxmax()
rank_changed = (final_comparison['Traditional Rank'] != final_comparison['Profit Rank']).any()

print('='*65)
print('                    CONCLUSION')
print('='*65)
print(f"""
This project evaluated three machine learning models —
Logistic Regression, Random Forest, and Gradient Boosting —
for customer churn prediction using two evaluation frameworks:
traditional metrics and profit-based metrics.

TRADITIONAL EVALUATION:
  Under accuracy-based metrics (Accuracy, Precision, Recall,
  F1, ROC-AUC), {best_trad} ranked first.
  Models were ranked purely on statistical performance without
  considering the financial impact of misclassifications.

PROFIT-BASED EVALUATION:
  Using a CLV-based cost-profit matrix and Expected Maximum
  Profit (EMP) across all classification thresholds,
  {best_profit} generated the highest total profit.
  The optimal threshold for each model differed from the
  default 0.5, confirming that threshold tuning is essential
  when business costs are asymmetric.

KEY FINDING:
  {'Model rankings CHANGED between traditional and profit evaluation.' if rank_changed
   else 'Rankings were consistent — but profit gaps between models were significant.'}
  A model with a higher AUC score does not necessarily
  produce the highest profit. This is because AUC treats
  all errors equally, while in reality a missed churner
  (False Negative) costs far more than a wasted retention
  offer (False Positive).

RESEARCH GAP ADDRESSED:
  This study empirically confirms the finding of Imani et al.
  (2025) — that most churn studies rely on accuracy-centric
  metrics. Our results demonstrate that profit-based evaluation
  leads to different and more business-aligned model selection.

PRACTICAL RECOMMENDATION:
  Organisations deploying churn models should evaluate using
  profit-based metrics and tune classification thresholds to
  reflect the actual cost of customer loss versus the cost
  of retention campaigns.
""")
print('='*65)

### 8.3 — Reproducibility Checklist

In [ ]:
import os

required_files = [
    'WA_Fn-UseC_-Telco-Customer-Churn.csv',
    'churn_cleaned.csv',
    'X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv',
    'logistic_regression_model.pkl',
    'random_forest_model.pkl',
    'gradient_boosting_model.pkl',
    'traditional_metrics.csv',
    'traditional_rankings.csv',
    'final_comparison.csv',
]

print('=== Reproducibility Checklist ===')
all_present = True
for f in required_files:
    exists = os.path.exists(f)
    status = '✓' if exists else '✗ MISSING'
    if not exists:
        all_present = False
    print(f'  {status}  {f}')

print()
if all_present:
    print('All files present — project is fully reproducible ✓')
else:
    print('WARNING: Some files are missing. Re-run the relevant step notebooks.')

In [ ]:
print('\n' + '='*65)
print('   PROJECT COMPLETE — ALL 8 STEPS FINISHED')
print('='*65)
print()
print('  Step 1  ✓  Data Collection & Understanding')
print('  Step 2  ✓  Data Cleaning & Preparation')
print('  Step 3  ✓  EDA & Visualization (6 plots)')
print('  Step 4  ✓  Preprocessing & Train/Test Split')
print('  Step 5  ✓  Model Training (3 models, 5-fold CV)')
print('  Step 6  ✓  Traditional Evaluation')
print('  Step 7  ✓  Profit-Based Evaluation (EMP)')
print('  Step 8  ✓  Conclusion & Documentation')
print()
print('  Total plots generated : 20')
print('  Models trained        : 3')
print('  Evaluation frameworks : 2 (Traditional + Profit)')
print()
print('  Presentation: 2nd March 2026')
print('='*65)